In [ ]:
# ================================================================
# 21/9 HOLD-OUT CALIBRATION + GRID-SEARCH OPTIMIZATION
# 2D-ADDS MODEL — ONE CELL / ONE COMMAND
# ================================================================
#
# WORKFLOW
# --------
# 30 reconstructed nodes
#        ↓
# reproducible 21/9 split
#        ↓
# 21 calibration nodes
#        ↓
# grid-search Q, D, lambda
#        ↓
# minimum calibration RMSE
#        ↓
# freeze Q, D, lambda
#        ↓
# 9 blind hold-out nodes
#
# FEATURES
# --------
# • Actual explicit FDM forward model
# • Forward Euler time integration
# • First-order upwind advection
# • Second-order central diffusion
# • Surface deposition + gravitational settling
# • Homogeneous Neumann boundary condition: du/dn = 0
# • dt fixed at 0.25 s, consistent with the manuscript
# • Exhaustive search over 11,840 prescribed combinations
# • Manual parameter test section
# • Side-by-side Optimized vs Manual_Test table
# • Relative RMSE-change diagnostics between successive evaluations/best solutions
# • Hold-out R², RMSE, MAE, MAPE and FAC2
# • Required plots
#
# IMPORTANT
# ---------
# The observations and coordinates below are demonstration values.
# Replace them with the ACTUAL 30 reconstructed observations and
# coordinates before reporting any calibration or validation result.
#
# ================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import product
from sklearn.metrics import r2_score

# ================================================================
# 1. 30 RECONSTRUCTED OBSERVATIONS AND NODE COORDINATES
# ================================================================

# DEMONSTRATION VALUES — REPLACE WITH ACTUAL DATA
observed = np.array([
    120, 145, 160, 180, 200,
    215, 230, 250, 270, 290,
    310, 330, 350, 370, 390,
    410, 430, 450, 470, 490,
    510, 530, 550, 570, 590,
    610, 630, 650, 670, 690
], dtype=float)

node_x = np.array([
     60,  80, 100, 120, 140,
     60,  80, 100, 120, 140,
     60,  80, 100, 120, 140,
     60,  80, 100, 120, 140,
     60,  80, 100, 120, 140,
     60,  80, 100, 120, 140
], dtype=float)

node_y = np.array([
     60,  60,  60,  60,  60,
     80,  80,  80,  80,  80,
    100, 100, 100, 100, 100,
    120, 120, 120, 120, 120,
    140, 140, 140, 140, 140,
    160, 160, 160, 160, 160
], dtype=float)

if len(observed) != 30 or len(node_x) != 30 or len(node_y) != 30:
    raise ValueError(
        "Exactly 30 observed values, 30 x-coordinates and 30 y-coordinates are required."
    )

# ================================================================
# 2. REPRODUCIBLE 21/9 SPLIT
# ================================================================

random_seed = 42
rng = np.random.default_rng(random_seed)

indices = np.arange(30)
rng.shuffle(indices)

calibration_idx = np.sort(indices[:21])
holdout_idx = np.sort(indices[21:])

observed_cal = observed[calibration_idx]
x_cal = node_x[calibration_idx]
y_cal = node_y[calibration_idx]

observed_holdout = observed[holdout_idx]
x_holdout = node_x[holdout_idx]
y_holdout = node_y[holdout_idx]

# ================================================================
# 3. FIXED MODEL / PHYSICAL PARAMETERS
# ================================================================

Lx, Ly = 200.0, 200.0
Nx, Ny = 101, 101

x = np.linspace(0.0, Lx, Nx)
y = np.linspace(0.0, Ly, Ny)

dx = x[1] - x[0]
dy = y[1] - y[0]

X, Y = np.meshgrid(x, y)

# Meteorology
vx = 2.30       # m/s
vy = 0.50       # m/s

# Atmospheric mixing height
H = 50.0        # m

# Particle properties
rho_particle = 1100.0      # kg/m^3
rho_air = 1.204            # kg/m^3
g = 9.81                    # m/s^2
particle_diameter = 75e-6   # m
mu = 1.81e-5                # Pa s

# Stokes settling velocity
Vs = (
    (rho_particle - rho_air)
    * g
    * particle_diameter**2
    / (18.0 * mu)
)

# Emission source
xf = 100.0
yf = 100.0
sigma = 2.5                 # m

# ================================================================
# 4. FIXED TIME STEP FROM THE MANUSCRIPT
# ================================================================
# IMPORTANT: dt is explicitly 0.25 s.
# It is NOT recalculated to 0.4 s or another value.

dt = 0.25                   # s
T = 120.0                   # s
steps = int(round(T / dt))
T_actual = steps * dt

# Stability checks for the largest searched D
D_max = 2.0

CFL_total = (
    abs(vx) * dt / dx
    + abs(vy) * dt / dy
)

# For explicit 2-D central diffusion, use:
# 2*D*dt/dx^2 + 2*D*dt/dy^2 <= 1
diffusion_number = (
    2.0 * D_max * dt / dx**2
    + 2.0 * D_max * dt / dy**2
)

if CFL_total > 1.0:
    raise ValueError(
        f"Advection CFL condition violated: CFL = {CFL_total:.6f}"
    )

if diffusion_number > 1.0:
    raise ValueError(
        f"Diffusion stability condition violated: "
        f"number = {diffusion_number:.6f}"
    )

# ================================================================
# 5. PARAMETER SEARCH SPACE
# ================================================================

Q_values = np.arange(20.0, 200.0 + 5.0, 5.0)
D_values = np.round(np.arange(0.5, 2.0 + 0.1, 0.1), 10)
lambda_values = np.round(
    np.arange(0.001, 0.020 + 0.001, 0.001),
    10
)

n_combinations = (
    len(Q_values)
    * len(D_values)
    * len(lambda_values)
)

# ================================================================
# 6. GAUSSIAN SOURCE SHAPE
# ================================================================

source_shape = np.exp(
    -(
        (X - xf)**2
        + (Y - yf)**2
    ) / (2.0 * sigma**2)
)

# ================================================================
# 7. MAP NODE COORDINATES TO THE COMPUTATIONAL GRID
# ================================================================

def node_indices(x_values, y_values):
    ix = np.abs(x[None, :] - x_values[:, None]).argmin(axis=1)
    iy = np.abs(y[None, :] - y_values[:, None]).argmin(axis=1)
    return iy, ix

cal_iy, cal_ix = node_indices(x_cal, y_cal)
hold_iy, hold_ix = node_indices(x_holdout, y_holdout)

# ================================================================
# 8. HOMOGENEOUS NEUMANN BOUNDARY CONDITION
# ================================================================
#
# ∂u/∂n = 0 at:
#   • inflow boundary
#   • lateral/open boundaries
#   • outflow boundary
#
# Discrete implementation:
#   left   : u[:,0]  = u[:,1]
#   right  : u[:,-1] = u[:,-2]
#   bottom : u[0,:]  = u[1,:]
#   top    : u[-1,:] = u[-2,:]

def apply_neumann_zero_gradient(u):
    u[:, 0] = u[:, 1]       # inflow
    u[:, -1] = u[:, -2]     # outflow
    u[0, :] = u[1, :]       # lateral/open
    u[-1, :] = u[-2, :]     # lateral/open
    return u

# ================================================================
# 9. ACTUAL 2D-ADDS FINITE-DIFFERENCE FORWARD MODEL
# ================================================================

def run_model(Q, D, lam):
    """
    2D-ADDS forward model.

    Time:
        Forward Euler

    Advection:
        First-order upwind

    Diffusion:
        Second-order central finite difference

    Removal:
        -(lambda + Vs/H) * u

    Source:
        Q * Gaussian spatial source

    Boundary:
        Homogeneous Neumann, du/dn = 0
    """

    u = np.zeros((Ny, Nx), dtype=float)

    source = Q * source_shape
    loss = lam + Vs / H

    for _ in range(steps):

        un = u.copy()

        # --------------------------------------------------------
        # First-order upwind advection
        # vx > 0 and vy > 0 for the present meteorological forcing.
        # --------------------------------------------------------
        dudx = (
            un[1:-1, 1:-1]
            - un[1:-1, :-2]
        ) / dx

        dudy = (
            un[1:-1, 1:-1]
            - un[:-2, 1:-1]
        ) / dy

        # --------------------------------------------------------
        # Second-order central diffusion
        # --------------------------------------------------------
        d2udx2 = (
            un[1:-1, 2:]
            - 2.0 * un[1:-1, 1:-1]
            + un[1:-1, :-2]
        ) / dx**2

        d2udy2 = (
            un[2:, 1:-1]
            - 2.0 * un[1:-1, 1:-1]
            + un[:-2, 1:-1]
        ) / dy**2

        # --------------------------------------------------------
        # Explicit Euler update
        # --------------------------------------------------------
        u_new = un.copy()

        u_new[1:-1, 1:-1] = (
            un[1:-1, 1:-1]
            + dt * (
                -vx * dudx
                -vy * dudy
                +D * d2udx2
                +D * d2udy2
                -loss * un[1:-1, 1:-1]
                +source[1:-1, 1:-1]
            )
        )

        # --------------------------------------------------------
        # Apply homogeneous Neumann boundaries
        # --------------------------------------------------------
        u_new = apply_neumann_zero_gradient(u_new)

        # Numerical safeguard
        u = np.maximum(u_new, 0.0)

    return u

# ================================================================
# 10. CALIBRATION OBJECTIVE FUNCTION
# ================================================================

def calibration_rmse(observed_values, predicted_values):
    """
    RMSE_cal =
    sqrt[(1/n_cal) * sum_k (P_k(theta) - O_k)^2]
    """

    observed_values = np.asarray(observed_values, dtype=float)
    predicted_values = np.asarray(predicted_values, dtype=float)

    if observed_values.shape != predicted_values.shape:
        raise ValueError(
            "Observed and predicted arrays must have identical shapes."
        )

    return float(
        np.sqrt(
            np.mean(
                (predicted_values - observed_values)**2
            )
        )
    )

# ================================================================
# 11. EXHAUSTIVE GRID SEARCH
# ================================================================
#
# Because this PDE is linear in Q, run Q=1 once for each (D, lambda)
# pair and scale the resulting predictions for all allowed Q values.
#
# All 11,840 parameter combinations are still evaluated by RMSE.

results = []

best_rmse = np.inf
best_Q = None
best_D = None
best_lambda = None
best_field = None

# RMSE-change diagnostics
# -----------------------
# These diagnostics monitor relative RMSE changes without changing
# the exhaustive grid-search procedure or its stopping behavior.
previous_evaluated_rmse = None
best_history = []
evaluation_counter = 0

print("=" * 78)
print("21-NODE CALIBRATION GRID SEARCH")
print("=" * 78)
print(f"Total parameter combinations : {n_combinations:,}")
print(f"Calibration nodes             : {len(calibration_idx)}")
print(f"Hold-out nodes                : {len(holdout_idx)}")
print(f"dt                            : {dt:.2f} s")
print(f"Final physical time           : {T_actual:.1f} s")
print(f"dx, dy                        : {dx:.2f}, {dy:.2f} m")
print(f"Vs                            : {Vs:.6e} m/s")
print(f"CFL total                     : {CFL_total:.6f}")
print(f"Diffusion number              : {diffusion_number:.6f}")
print("Boundary condition            : du/dn = 0")
print("=" * 78)

pair_total = len(D_values) * len(lambda_values)
pair_counter = 0

for D, lam in product(D_values, lambda_values):

    field_Q1 = run_model(1.0, D, lam)
    predicted_Q1 = field_Q1[cal_iy, cal_ix]

    for Q in Q_values:

        predicted_cal = Q * predicted_Q1

        rmse_cal = calibration_rmse(
            observed_cal,
            predicted_cal
        )

        evaluation_counter += 1

        # Relative change in RMSE between successive parameter evaluations.
        # This is a diagnostic only; it does not alter the exhaustive search.
        if previous_evaluated_rmse is None:
            rmse_change_pct = np.nan
        else:
            rmse_change_pct = (
                abs(rmse_cal - previous_evaluated_rmse)
                / max(abs(previous_evaluated_rmse), np.finfo(float).eps)
            ) * 100.0

        results.append({
            "Iteration": int(evaluation_counter),
            "Q": float(Q),
            "D": float(D),
            "lambda": float(lam),
            "RMSE_cal": float(rmse_cal),
            "RMSE_change_from_previous_%": float(rmse_change_pct)
        })

        if rmse_cal < best_rmse:
            previous_best_rmse = best_rmse

            best_rmse = float(rmse_cal)
            best_Q = float(Q)
            best_D = float(D)
            best_lambda = float(lam)
            best_field = Q * field_Q1

            if np.isfinite(previous_best_rmse):
                best_rmse_change_pct = (
                    abs(best_rmse - previous_best_rmse)
                    / max(abs(previous_best_rmse), np.finfo(float).eps)
                ) * 100.0
            else:
                best_rmse_change_pct = np.nan

            best_history.append({
                "Iteration": int(evaluation_counter),
                "Q": float(best_Q),
                "D": float(best_D),
                "lambda": float(best_lambda),
                "Best_RMSE": float(best_rmse),
                "Relative_change_from_previous_best_%":
                    float(best_rmse_change_pct)
            })

        previous_evaluated_rmse = float(rmse_cal)

    pair_counter += 1

    if pair_counter % 20 == 0 or pair_counter == pair_total:
        print(
            f"Completed {pair_counter:>3}/{pair_total} "
            f"(D, lambda) pairs | "
            f"Best RMSE = {best_rmse:.6f}"
        )

results_df = pd.DataFrame(results).sort_values(
    "RMSE_cal",
    ascending=True
).reset_index(drop=True)

# Verify best calibration result
best_predictions_cal = best_field[cal_iy, cal_ix]
best_rmse_verified = calibration_rmse(
    observed_cal,
    best_predictions_cal
)

# ================================================================
# 11A. RMSE RELATIVE-CHANGE DIAGNOSTICS
# ================================================================
#
# The exhaustive grid search remains unchanged: all prescribed
# parameter combinations are evaluated. The relative RMSE change is
# reported only as a numerical diagnostic.
#
# For successive parameter evaluations:
#   Relative change (%) =
#   100 * |RMSE_i - RMSE_(i-1)| / |RMSE_(i-1)|
#
# For successive "best-so-far" solutions, the same expression is
# applied whenever a new lower RMSE is found.
# ================================================================

best_history_df = pd.DataFrame(best_history)

if not best_history_df.empty:
    print("\n" + "=" * 78)
    print("SUCCESSIVE BEST-SOLUTION RMSE CHANGE")
    print("=" * 78)
    display(
        best_history_df.tail(10).style.format({
            "Q": "{:.6f}",
            "D": "{:.6f}",
            "lambda": "{:.6f}",
            "Best_RMSE": "{:.6f}",
            "Relative_change_from_previous_best_%": "{:.6f}"
        })
    )

    finite_changes = best_history_df[
        "Relative_change_from_previous_best_%"
    ].dropna()

    if len(finite_changes) > 0:
        final_best_change_pct = float(finite_changes.iloc[-1])
        print(
            f"Relative RMSE change from the previous best solution "
            f"to the final best solution: {final_best_change_pct:.6f}%"
        )
        print(
            "Diagnostic threshold (manuscript criterion): 0.1%"
        )
        if final_best_change_pct < 0.1:
            print(
                "The final improvement in the best-so-far RMSE is below "
                "0.1%. This is reported as a diagnostic only; the exhaustive "
                "grid search is still completed."
            )
        else:
            print(
                "The final improvement in the best-so-far RMSE is not below "
                "0.1%. The exhaustive grid search is still completed."
            )

# ================================================================
# 12. MANUAL PARAMETER TEST — EDIT THESE THREE VALUES
# ================================================================
#
# ENTER ANY VALUES WITHIN THE PRESCRIBED SEARCH BOUNDS HERE.
#
# Example:
# Q_manual      = 100.0
# D_manual      = 1.0
# lambda_manual = 0.005
#
# The same FDM model is run and its calibration RMSE is calculated
# over the SAME 21 calibration observations.
# ================================================================

Q_manual = 100.0
D_manual = 1.0
lambda_manual = 0.005

if not (20.0 <= Q_manual <= 200.0):
    raise ValueError(
        "Q_manual must be between 20 and 200 particles/s."
    )

if not (0.5 <= D_manual <= 2.0):
    raise ValueError(
        "D_manual must be between 0.5 and 2.0 m²/s."
    )

if not (0.001 <= lambda_manual <= 0.020):
    raise ValueError(
        "lambda_manual must be between 0.001 and 0.020 s⁻¹."
    )

manual_field = run_model(
    Q_manual,
    D_manual,
    lambda_manual
)

manual_predictions_cal = manual_field[cal_iy, cal_ix]

manual_rmse = calibration_rmse(
    observed_cal,
    manual_predictions_cal
)

# ================================================================
# 13. OPTIMIZED VS MANUAL_TEST TABLE
# ================================================================
#
# This produces a table in the requested structure:
#
# |   | Parameter | Optimized | Manual_Test |
# | 0 | Q         | ...       | ...         |
# | 1 | D         | ...       | ...         |
# | 2 | lambda    | ...       | ...         |
# | 3 | RMSE      | ...       | ...         |

comparison = pd.DataFrame({
    "Parameter": [
        "Q (particles/s)",
        "D (m²/s)",
        "lambda (s⁻¹)",
        "Calibration RMSE"
    ],
    "Optimized": [
        best_Q,
        best_D,
        best_lambda,
        best_rmse_verified
    ],
    "Manual_Test": [
        Q_manual,
        D_manual,
        lambda_manual,
        manual_rmse
    ]
})

print("\n" + "=" * 78)
print("OPTIMIZED VS MANUAL PARAMETER TEST")
print("=" * 78)

display(
    comparison.style.format({
        "Optimized": "{:.6f}",
        "Manual_Test": "{:.6f}"
    })
)

# RMSE difference between the manual and optimized parameter sets.
# Positive values mean the manual-test RMSE is higher than the optimized RMSE.
rmse_difference = float(manual_rmse - best_rmse_verified)
rmse_relative_change_vs_optimized = (
    abs(rmse_difference)
    / max(abs(best_rmse_verified), np.finfo(float).eps)
) * 100.0

rmse_comparison = pd.DataFrame({
    "Metric": [
        "Absolute RMSE difference (Manual − Optimized)",
        "Relative RMSE change vs Optimized (%)"
    ],
    "Value": [
        rmse_difference,
        rmse_relative_change_vs_optimized
    ]
})

print("\nRMSE CHANGE: OPTIMIZED VS MANUAL TEST")
display(
    rmse_comparison.style.format({
        "Value": "{:.6f}"
    })
)

print("=" * 78)

if manual_rmse < best_rmse_verified - 1e-12:
    print(
        "WARNING: Manual-test RMSE is lower than the exhaustive-search "
        "minimum. Check the search grid and implementation."
    )
elif np.isclose(
    manual_rmse,
    best_rmse_verified,
    rtol=1e-10,
    atol=1e-10
):
    print(
        "The manual parameter set matches the minimum calibration RMSE "
        "within numerical tolerance."
    )
else:
    print(
        f"The optimized RMSE is lower than the manual-test RMSE by "
        f"{manual_rmse - best_rmse_verified:.6f} particles."
    )

# ================================================================
# 14. DISPLAY THE BEST PARAMETER VALUES
# ================================================================

print("\n" + "=" * 78)
print("BEST PARAMETER VECTOR FOR MINIMUM CALIBRATION RMSE")
print("=" * 78)
print(f"Q       = {best_Q:.6f} particles/s")
print(f"D       = {best_D:.6f} m²/s")
print(f"lambda  = {best_lambda:.6f} s⁻¹")
print(f"RMSE    = {best_rmse_verified:.6f} particles")
print("=" * 78)

print("\nTOP 10 CALIBRATION PARAMETER COMBINATIONS")
display(
    results_df.head(10).style.format({
        "Iteration": "{:.0f}",
        "Q": "{:.6f}",
        "D": "{:.6f}",
        "lambda": "{:.6f}",
        "RMSE_cal": "{:.6f}",
        "RMSE_change_from_previous_%": "{:.6f}"
    })
)

# ================================================================
# 15. FREEZE OPTIMIZED PARAMETERS
# ================================================================

theta_hat = (
    float(best_Q),
    float(best_D),
    float(best_lambda)
)

# ================================================================
# 16. 9-NODE BLIND HOLD-OUT PREDICTIONS
# ================================================================
# IMPORTANT:
# The 9 hold-out observations are NOT used to select theta_hat.

predicted_holdout = best_field[hold_iy, hold_ix]

# ================================================================
# 17. HOLD-OUT PERFORMANCE METRICS
# ================================================================

def calculate_metrics(observed_values, predicted_values):

    observed_values = np.asarray(observed_values, dtype=float)
    predicted_values = np.asarray(predicted_values, dtype=float)

    r2 = r2_score(
        observed_values,
        predicted_values
    )

    rmse = np.sqrt(
        np.mean(
            (predicted_values - observed_values)**2
        )
    )

    mae = np.mean(
        np.abs(
            predicted_values - observed_values
        )
    )

    nonzero = observed_values != 0

    if np.any(nonzero):
        mape = 100.0 * np.mean(
            np.abs(
                (predicted_values[nonzero] - observed_values[nonzero])
                / observed_values[nonzero]
            )
        )
    else:
        mape = np.nan

    ratio = (
        predicted_values[nonzero]
        / observed_values[nonzero]
    )

    fac2 = np.mean(
        (ratio >= 0.5)
        & (ratio <= 2.0)
    )

    return r2, rmse, mae, mape, fac2

r2_val, rmse_val, mae_val, mape_val, fac2_val = calculate_metrics(
    observed_holdout,
    predicted_holdout
)

print("\n" + "=" * 78)
print("9-NODE BLIND HOLD-OUT PERFORMANCE")
print("=" * 78)
print(f"R²   = {r2_val:.6f}")
print(f"RMSE = {rmse_val:.6f} particles")
print(f"MAE  = {mae_val:.6f} particles")
print(f"MAPE = {mape_val:.6f}%")
print(f"FAC2 = {fac2_val:.6f} ({fac2_val*100:.2f}%)")
print("=" * 78)

# ================================================================
# 18. HOLD-OUT TABLE
# ================================================================

holdout_table = pd.DataFrame({
    "Node": holdout_idx + 1,
    "x_m": x_holdout,
    "y_m": y_holdout,
    "Observed": observed_holdout,
    "Predicted": predicted_holdout,
    "Residual_Observed_minus_Predicted":
        observed_holdout - predicted_holdout
})

display(
    holdout_table.style.format({
        "x_m": "{:.3f}",
        "y_m": "{:.3f}",
        "Observed": "{:.4f}",
        "Predicted": "{:.4f}",
        "Residual_Observed_minus_Predicted": "{:.4f}"
    })
)

# ================================================================
# 19. FIGURE 1 — GRID-SEARCH OBJECTIVE FUNCTION
# ================================================================

plt.figure(figsize=(8, 5))

plt.plot(
    np.arange(1, len(results_df) + 1),
    results_df["RMSE_cal"].values,
    linewidth=1.5
)

plt.axhline(
    best_rmse_verified,
    linestyle="--",
    linewidth=1.5,
    label=f"Minimum RMSE = {best_rmse_verified:.2f}"
)

plt.xlabel(
    "Parameter combination ranked by calibration RMSE"
)
plt.ylabel(
    "Calibration RMSE (particles)"
)
plt.title(
    "Grid-Search Calibration Objective Function"
)
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

# ================================================================
# 20. FIGURE 2 — HOLD-OUT OBSERVED VS PREDICTED
# ================================================================

plt.figure(figsize=(7, 7))

plt.scatter(
    observed_holdout,
    predicted_holdout,
    s=100,
    alpha=0.8,
    label="Blind hold-out nodes"
)

xmin = min(
    observed_holdout.min(),
    predicted_holdout.min()
)

xmax = max(
    observed_holdout.max(),
    predicted_holdout.max()
)

plt.plot(
    [xmin, xmax],
    [xmin, xmax],
    linestyle="--",
    linewidth=2,
    label="1:1 Perfect Agreement"
)

metrics_text = (
    rf"$R^2 = {r2_val:.2f}$" "\n"
    rf"$RMSE = {rmse_val:.1f}$ particles" "\n"
    rf"$MAE = {mae_val:.1f}$ particles" "\n"
    rf"$MAPE = {mape_val:.2f}\%$" "\n"
    rf"$FAC2 = {fac2_val:.2f}$"
)

plt.text(
    0.05,
    0.95,
    metrics_text,
    transform=plt.gca().transAxes,
    fontsize=12,
    verticalalignment="top",
    bbox=dict(
        boxstyle="round,pad=0.5",
        facecolor="white",
        alpha=0.85
    )
)

plt.xlabel(
    "Observed Microplastic Counts (particles)",
    fontsize=13
)
plt.ylabel(
    "Model-Predicted Microplastic Counts (particles)",
    fontsize=13
)
plt.title(
    "Blind Hold-Out Evaluation: Observed vs. Model-Predicted",
    fontsize=14
)
plt.grid(True, linestyle=":", alpha=0.6)
plt.legend()
plt.tight_layout()
plt.show()

# ================================================================
# 21. FIGURE 3 — HOLD-OUT RESIDUALS
# ================================================================

residual_holdout = (
    observed_holdout -
    predicted_holdout
)

plt.figure(figsize=(8, 5))

plt.axhline(
    0,
    linestyle="--",
    linewidth=1.5
)

plt.scatter(
    holdout_idx + 1,
    residual_holdout,
    s=90,
    alpha=0.8
)

plt.xlabel(
    "Hold-Out Node",
    fontsize=13
)
plt.ylabel(
    "Residual (Observed − Predicted)",
    fontsize=13
)
plt.title(
    "Residuals for the 9 Blind Hold-Out Nodes",
    fontsize=14
)
plt.grid(True, linestyle=":", alpha=0.6)
plt.tight_layout()
plt.show()

# ================================================================
# 22. FIGURE 4 — CALIBRATED CONCENTRATION FIELD
# ================================================================

plt.figure(figsize=(8, 6))

contour = plt.contourf(
    X,
    Y,
    best_field,
    levels=40
)

plt.colorbar(
    contour,
    label="Model-predicted concentration"
)

plt.scatter(
    xf,
    yf,
    s=90,
    marker="*",
    label="Emission source"
)

plt.scatter(
    x_cal,
    y_cal,
    s=30,
    marker="o",
    label="Calibration nodes"
)

plt.scatter(
    x_holdout,
    y_holdout,
    s=70,
    facecolors="none",
    edgecolors="black",
    label="Blind hold-out nodes"
)

plt.xlabel("x (m)")
plt.ylabel("y (m)")
plt.title(
    "Calibrated 2D-ADDS Concentration Field"
    + f"\nQ={best_Q:.1f}, D={best_D:.2f}, "
    + rf"$\lambda$={best_lambda:.3f}, t={T_actual:.1f} s"
)
plt.legend()
plt.tight_layout()
plt.show()

# ================================================================
# 23. FINAL INTERPRETATION
# ================================================================

print("\n" + "=" * 78)
print("FINAL INTERPRETATION")
print("=" * 78)
print(
    "The optimized Q, D and lambda minimize calibration RMSE over "
    "the 21 calibration nodes."
)
print(
    "The Manual_Test values can be changed in Section 12 to test "
    "different parameter combinations and their calibration RMSE."
)
print(
    "Relative RMSE changes are reported as diagnostics and do not "
    "terminate the exhaustive grid search."
)
print(
    "The optimized parameter vector is frozen before the 9-node "
    "hold-out evaluation."
)
print(
    "The hold-out observations are not used during calibration."
)
print(
    "The FDM time step is fixed at dt = 0.25 s, consistent with "
    "the manuscript."
)
print(
    "The boundary condition is homogeneous Neumann, du/dn = 0, "
    "at inflow, lateral/open boundaries and outflow."
)
print(
    "Replace demonstration observations and coordinates with the "
    "actual reconstructed secondary dataset before reporting results."
)
print("=" * 78)

# ================================================================
# END — RUN THIS ENTIRE CELL ONCE
# ================================================================
